# Chapter 05: Geometric Deep Learning Models

**Source orientation.** Printed pages 68-101; physical PDF pages 72-105. This notebook is an original, executable lesson built from the chapter's concepts and structure. It does not reproduce textbook prose, exercises, figures, screenshots, or page crops.

**Chapter question.** CNNs, group CNNs, GNNs, sets/Transformers, E(3) message passing, mesh CNNs, and RNN/LSTM.

## Route Through The Notebook

- Translate the chapter's core ideas into domain, transformation, representation, and check vocabulary.
- Generate visual artifacts under `artifacts/chapter-05/` and display them inline.
- Record numerical residuals or structural checks in a JSON ledger.
- End with a small lab prompt so the reader can test the geometric assumption.


In [ ]:
# geometry-setup:v1
# Machine-managed by scripts/update_notebook_setup.py. Do not edit this cell by hand.

from __future__ import annotations

import json as _geometry_json
import os as _geometry_os
from pathlib import Path as _GeometryPath
import sys as _geometry_sys

GEOMETRY_SETUP = _geometry_json.loads(
    r"""
{
  "colab_url": "https://colab.research.google.com/github/Rah-Rah-Mitra/Geometry/blob/main/Geometric-Deep-Learning/chapter-05-geometric-deep-learning-models/05-geometric-deep-learning-models.ipynb",
  "course_dir": "Geometric-Deep-Learning",
  "course_title": "Geometric Deep Learning",
  "github_url": "https://github.com/Rah-Rah-Mitra/Geometry/blob/main/Geometric-Deep-Learning/chapter-05-geometric-deep-learning-models/05-geometric-deep-learning-models.ipynb",
  "jupyterlite": false,
  "marker": "geometry-setup:v1",
  "notebook_kind": "lesson",
  "notebook_path": "Geometric-Deep-Learning/chapter-05-geometric-deep-learning-models/05-geometric-deep-learning-models.ipynb",
  "notebook_title": "Chapter 05: Geometric Deep Learning Models",
  "repository": {
    "branch": "main",
    "name": "Geometry",
    "owner": "Rah-Rah-Mitra",
    "source_url": "https://github.com/Rah-Rah-Mitra/Geometry"
  },
  "requirements": "requirements/ml-geometry.txt",
  "runtime_profile": "ml_geometry"
}
"""
)


def _geometry_is_colab():
    try:
        import google.colab  # type: ignore  # noqa: F401
        return True
    except Exception:
        return False


def _geometry_is_jupyterlite():
    return _geometry_sys.platform == "emscripten" or "pyodide" in _geometry_sys.modules


def _geometry_add_path(path):
    text = str(path)
    if text not in _geometry_sys.path:
        _geometry_sys.path.insert(0, text)


def _geometry_find_repo_root():
    candidates = []
    env_root = _geometry_os.environ.get("GEOMETRY_REPO_ROOT")
    if env_root:
        candidates.append(_GeometryPath(env_root).expanduser())
    candidates.append(_GeometryPath.cwd())
    for start in candidates:
        start = start.resolve()
        for current in (start, *start.parents):
            if (current / "course-manifest.json").exists() and (
                current / "metadata" / "runtime_profiles.yml"
            ).exists():
                return current
    raise RuntimeError(
        "Could not find the Geometry repository root. Start JupyterLab inside the "
        "Geometry checkout or set GEOMETRY_REPO_ROOT."
    )


def _geometry_run(command):
    import subprocess as _geometry_subprocess

    printable = " ".join(str(part) for part in command)
    print(f"+ {printable}")
    _geometry_subprocess.check_call([str(part) for part in command])


def _geometry_requirement_names(requirements_path, seen=None):
    seen = set() if seen is None else seen
    requirements_path = requirements_path.resolve()
    if requirements_path in seen or not requirements_path.exists():
        return []
    seen.add(requirements_path)
    names = []
    for raw_line in requirements_path.read_text(encoding="utf-8").splitlines():
        line = raw_line.split("#", 1)[0].strip()
        if not line:
            continue
        if line.startswith(("-r ", "--requirement ")):
            _, nested = line.split(maxsplit=1)
            names.extend(_geometry_requirement_names(requirements_path.parent / nested, seen))
            continue
        if line.startswith("-"):
            continue
        name = line
        for separator in ("==", ">=", "<=", "~=", "!=", ">", "<", ";"):
            name = name.split(separator, 1)[0]
        name = name.split("[", 1)[0].strip()
        if name:
            names.append(name)
    return sorted(set(names))


def _geometry_missing_requirements(requirements_path):
    import importlib.metadata as _geometry_metadata

    missing = []
    for name in _geometry_requirement_names(requirements_path):
        try:
            _geometry_metadata.distribution(name)
        except _geometry_metadata.PackageNotFoundError:
            missing.append(name)
    return missing


def _geometry_configured_roots(repo_root):
    course_dir = GEOMETRY_SETUP.get("course_dir")
    course_root = repo_root / course_dir if course_dir else repo_root
    return repo_root, course_root


if _geometry_is_jupyterlite():
    if not GEOMETRY_SETUP["jupyterlite"]:
        raise RuntimeError(
            "This Geometry notebook uses runtime profile "
            f"{GEOMETRY_SETUP['runtime_profile']!r}, which is not enabled for "
            "JupyterLite in course-manifest.json. Open it in Colab or local JupyterLab."
        )
    GEOMETRY_REPO_ROOT = _GeometryPath.cwd()
    GEOMETRY_COURSE_ROOT = (
        GEOMETRY_REPO_ROOT / GEOMETRY_SETUP["course_dir"]
        if GEOMETRY_SETUP.get("course_dir")
        else GEOMETRY_REPO_ROOT
    )
    _geometry_add_path(GEOMETRY_REPO_ROOT)
    if GEOMETRY_COURSE_ROOT.exists():
        _geometry_add_path(GEOMETRY_COURSE_ROOT)
    GEOMETRY_RUNTIME_PROFILE = GEOMETRY_SETUP["runtime_profile"]
    print(
        "Geometry setup: JupyterLite/Pyodide detected; shell, git, and pip steps "
        "were skipped."
    )
elif _geometry_is_colab():
    repository = GEOMETRY_SETUP["repository"]
    repo_url = repository["source_url"].rstrip("/") + ".git"
    branch = repository["branch"]
    GEOMETRY_REPO_ROOT = _GeometryPath(
        _geometry_os.environ.get("GEOMETRY_REPO_ROOT", "/content/Geometry")
    )
    sparse_paths = ["requirements", "metadata", "scripts", "course-manifest.json", "index.ipynb"]
    if GEOMETRY_SETUP.get("course_dir"):
        sparse_paths.append(GEOMETRY_SETUP["course_dir"])
    if not (GEOMETRY_REPO_ROOT / ".git").exists():
        if GEOMETRY_REPO_ROOT.exists() and any(GEOMETRY_REPO_ROOT.iterdir()):
            raise RuntimeError(
                f"{GEOMETRY_REPO_ROOT} exists but is not a git checkout. "
                "Set GEOMETRY_REPO_ROOT to an empty path or remove the directory."
            )
        _geometry_run(
            [
                "git",
                "clone",
                "--filter=blob:none",
                "--no-checkout",
                "--branch",
                branch,
                repo_url,
                GEOMETRY_REPO_ROOT,
            ]
        )
        _geometry_run(["git", "-C", GEOMETRY_REPO_ROOT, "sparse-checkout", "init", "--cone"])
    _geometry_run(["git", "-C", GEOMETRY_REPO_ROOT, "sparse-checkout", "set", *sparse_paths])
    _geometry_run(["git", "-C", GEOMETRY_REPO_ROOT, "checkout", branch])
    requirements_path = GEOMETRY_REPO_ROOT / GEOMETRY_SETUP["requirements"]
    _geometry_run([_geometry_sys.executable, "-m", "pip", "install", "-q", "-r", requirements_path])
    GEOMETRY_REPO_ROOT, GEOMETRY_COURSE_ROOT = _geometry_configured_roots(GEOMETRY_REPO_ROOT)
    _geometry_os.chdir(GEOMETRY_COURSE_ROOT if GEOMETRY_COURSE_ROOT.exists() else GEOMETRY_REPO_ROOT)
    _geometry_add_path(GEOMETRY_REPO_ROOT)
    _geometry_add_path(GEOMETRY_COURSE_ROOT)
    GEOMETRY_RUNTIME_PROFILE = GEOMETRY_SETUP["runtime_profile"]
    print(
        f"Geometry setup: Colab ready at {_GeometryPath.cwd()} "
        f"with profile {GEOMETRY_RUNTIME_PROFILE!r}."
    )
else:
    GEOMETRY_REPO_ROOT = _geometry_find_repo_root()
    requirements_path = GEOMETRY_REPO_ROOT / GEOMETRY_SETUP["requirements"]
    missing = _geometry_missing_requirements(requirements_path)
    skip_install = _geometry_os.environ.get("GEOMETRY_SKIP_INSTALL") == "1"
    if missing and skip_install:
        print(
            "Geometry setup: GEOMETRY_SKIP_INSTALL=1, so missing profile packages "
            f"were not installed: {', '.join(missing)}"
        )
    elif missing:
        print(
            "Geometry setup: installing missing profile packages from "
            f"{requirements_path.relative_to(GEOMETRY_REPO_ROOT)}: {', '.join(missing)}"
        )
        _geometry_run([_geometry_sys.executable, "-m", "pip", "install", "-r", requirements_path])
    GEOMETRY_REPO_ROOT, GEOMETRY_COURSE_ROOT = _geometry_configured_roots(GEOMETRY_REPO_ROOT)
    _geometry_os.chdir(GEOMETRY_COURSE_ROOT if GEOMETRY_COURSE_ROOT.exists() else GEOMETRY_REPO_ROOT)
    _geometry_add_path(GEOMETRY_REPO_ROOT)
    _geometry_add_path(GEOMETRY_COURSE_ROOT)
    GEOMETRY_RUNTIME_PROFILE = GEOMETRY_SETUP["runtime_profile"]
    print(
        f"Geometry setup: local checkout ready at {_GeometryPath.cwd()} "
        f"with profile {GEOMETRY_RUNTIME_PROFILE!r}."
    )


In [ ]:

from pathlib import Path
import sys

BOOK_ROOT = Path.cwd()
for candidate in [BOOK_ROOT, *BOOK_ROOT.parents]:
    if (candidate / '00-book-index.ipynb').exists() and (candidate / 'utils').exists():
        BOOK_ROOT = candidate
        break
else:
    raise RuntimeError('Could not find the Geometric Deep Learning course root')

if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

import pandas as pd
from IPython.display import display
from utils.artifacts import assert_artifacts, display_artifact, save_json
from utils.course_visuals import build_chapter_visuals
from utils.notebook_checks import file_size, image_nonblank

CHAPTER = 5
print(f'Book root: {BOOK_ROOT}')


## Translation Guide

The model chapter shows that common neural architectures instantiate choices about domain, symmetry, locality, and readout.

CNNs use translations on grids, GNNs use permutation-equivariant neighborhood aggregation, attention behaves like a learned soft graph, and E(3) models separate invariant distances from equivariant coordinates.

The visuals are deliberately small because the architectural promises can be tested before any large benchmark is involved.

The code cells below use shared helpers from `utils/` rather than hidden external assets. Every visual is regenerated from synthetic data so the chapter remains portable and inspectable. The checks are intentionally small but they target the same invariants that larger systems rely on: permutation equivariance, rigid-motion invariance, translation equivariance, spectral consistency, or graph-structured information flow.

## Working Vocabulary

- `domain`: the object that indexes the signal, such as a grid, graph, group, point cloud, surface, or sequence.
- `signal`: the values living on the domain.
- `transformation`: a change of coordinates or domain elements whose effect should be predictable.
- `invariant`: a quantity that should not change after an allowed transformation.
- `equivariant`: an output that should transform in the same organized way as the input.
- `artifact`: a generated figure, HTML view, table, or JSON file saved under the course-local artifact tree.


In [ ]:

ARTIFACTS, CHECKS = build_chapter_visuals(CHAPTER)
print(f'Generated {len(ARTIFACTS)} artifacts for chapter {CHAPTER:02d}')
for path in ARTIFACTS:
    print(path.relative_to(BOOK_ROOT))



## Standalone Study Notes

This notebook is written to be useful without the PDF open. The page span above is source orientation, not a reading dependency. Definitions are restated in fresh language, each visual is generated from code in this course, and the final ledger records the checks that make the lesson reproducible. The important habit is to translate every informal architecture claim into a transformation claim: what object changes, what representation changes, and what quantity should stay fixed or move predictably.

The words invariant, equivariant, stable, and local are kept separate throughout the course. An invariant scalar is expected to stay the same after an allowed transformation. An equivariant output is expected to transform in the corresponding way. A stable output is allowed to change, but only in proportion to a small deformation. A local computation restricts information flow to nearby elements before later layers or pooling enlarge the receptive field. These ideas are close enough to blur in prose, so the notebooks attach them to concrete residuals.

The examples are intentionally small. A tiny graph, a toy image, a short sequence, or a synthetic point cloud is easier to inspect than a benchmark-scale dataset, and the geometric claim is the same. When a residual is numerically zero, the construction has the advertised symmetry on that finite example. When a value is merely monotone or small, the notebook labels it as an empirical stability check rather than a proof. This distinction is part of the teaching contract.

A productive lab exercise is to break one assumption and rerun the notebook. Relabel graph nodes without relabeling the adjacency matrix, rotate coordinates but forget to rotate vector features, replace circular padding with zero padding, or change a local tangent frame without transporting coordinates. The failure will usually be visible before it is numerically dramatic. Visualization-first work is valuable because it catches conceptual bugs early.

The course also separates source orientation from authorship. The textbook provides chapter order, concepts, and notation, but this notebook supplies original explanations, examples, diagrams, and sanity checks. No textbook figures, screenshots, page crops, long exercise text, or copied passages are used. The goal is a standalone computational course: readable as prose, executable as notebooks, and auditable as a collection of artifacts.


In [ ]:

# Display the primary static visuals inline.
for path in ARTIFACTS:
    if path.suffix.lower() in {'.png', '.jpg', '.jpeg', '.gif', '.webp', '.svg'}:
display_artifact(path, width=760)


In [ ]:

# Display interactive HTML artifacts and link structured outputs.
for path in ARTIFACTS:
    if path.suffix.lower() in {'.html', '.htm'}:
display_artifact(path, height=580)
    elif path.suffix.lower() in {'.json', '.csv'}:
display_artifact(path)


In [ ]:

summary = assert_artifacts(ARTIFACTS)
for path in ARTIFACTS:
    if path.suffix.lower() == '.png':
        image_nonblank(path)
    else:
        file_size(path)

if CHAPTER == 1:
    assert CHECKS['grid_convolution_equivariance_error'] < 1e-12
    assert CHECKS['distance_invariance_error'] < 1e-12
    assert CHECKS['taxonomy_orphan_nodes'] == 0
elif CHAPTER == 2:
    assert max(CHECKS['interpolant_train_errors'].values()) < 1e-8
    assert CHECKS['minimum_norm_error'] < 1e-3
elif CHAPTER == 3:
    assert CHECKS['d3_noncommutative'] and CHECKS['d3_closed']
    assert CHECKS['conv_equivariance_error'] < 1e-12
    assert CHECKS['pooling_invariance_error'] < 1e-12
elif CHAPTER == 4:
    assert CHECKS['graph_equivariance_error'] < 1e-12
    assert CHECKS['dft_offdiag_residual'] < 1e-10
    assert CHECKS['sphere_target_norm_error'] < 1e-12
elif CHAPTER == 5:
    assert CHECKS['cnn_translation_equivariance_error'] < 1e-12
    assert CHECKS['gnn_permutation_equivariance_error'] < 1e-12
    assert CHECKS['e3_distance_preservation_error'] < 1e-12
elif CHAPTER == 6:
    assert CHECKS['molecule_distance_error'] < 1e-12
    assert CHECKS['sequence_attention_row_error'] < 1e-12
    assert CHECKS['particle_count'] == 80
elif CHAPTER == 7:
    assert CHECKS['timeline_event_count'] >= 8
    assert CHECKS['lineage_dag']
    assert CHECKS['wl_histograms_match']

ledger = {'chapter': CHAPTER, 'checks': CHECKS, 'artifact_sizes': summary}
ledger_path = save_json(ledger, CHAPTER, f'chapter-{CHAPTER:02d}-notebook-ledger.json')
display(pd.DataFrame([{'check': key, 'value': str(value)[:100]} for key, value in CHECKS.items()]))
print(f'Notebook ledger: {ledger_path.relative_to(BOOK_ROOT)}')


## Deeper Reading Notes

This chapter's theme is CNNs, group CNNs, GNNs, attention, equivariant point clouds, meshes, and sequences. The notebook treats that theme as a sequence of design decisions rather than a list of facts. First decide what the mathematical object is: a set of samples, a graph of relations, a grid with ordered neighbors, a point cloud with distances, a surface with tangent frames, or a sequence with temporal order. Then decide which changes of representation are merely coordinate changes and which changes should alter the answer. This separation is the main source of clarity in geometric deep learning.

The visuals should be read as arguments. A matrix heatmap is not only a picture of numbers; it shows whether relabeling, shifting, or diagonalizing behaves as claimed. A graph drawing is not an appeal to intuition alone; it pairs with a residual that checks permutation behavior. A point cloud view is not a molecular simulation; it is a controlled example where rigid motion preserves distances. The artifact is therefore both a teaching object and a small unit test for the concept.

When moving from these toy examples to real research code, the same questions remain useful. What transformations are built in exactly? What transformations are only approximated by data augmentation or optimization? Which quantities are invariant readouts, and which are equivariant intermediate fields? Where does locality enter, and how does pooling or message passing enlarge the scale? Which coordinate choices are arbitrary, and where must the model transport features between frames? Answering these questions often exposes whether an architecture matches the data before any training curve is plotted.

The chapter also has limits. The notebooks do not claim that a small synthetic example proves a theorem about large neural networks. They demonstrate the mechanism that a theorem, architecture, or experiment would later formalize at scale. That is why the final checks are intentionally narrow: they verify the invariant being taught, not every possible property of the method. A reader can extend the lab by replacing the toy domain with a larger one while keeping the same residuals.

A final way to use the notebook is as a vocabulary bridge. If a paper says equivariant, look for the representation and the commuting diagram. If it says invariant, look for the group action and the readout. If it says stable, look for a deformation metric and a bound or empirical sweep. If it says geometric, ask which domain structure is actually being used. This discipline keeps the word geometry from becoming decorative.



## Applied Lab And Takeaways

The applied lab is to make a controlled change to the geometric assumption and watch the checks respond. Change a permutation, modify a graph edge, alter the padding convention, rotate a point cloud, or perturb a mesh. The purpose is not to make every model invariant to every transformation. The purpose is to know which transformations the task should respect and which transformations carry semantic information.

Takeaways:

- Geometric deep learning starts from domains and transformations before it starts from layer names.
- Invariance, equivariance, stability, and locality are executable claims in these notebooks.
- The same blueprint explains many architectures while preserving their differences.
- Artifacts are part of the learning product; the final JSON ledger turns visual claims into auditable checks.
